In [25]:
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sbn
import numpy as np 

import torch

import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.metrics import silhouette_score, silhouette_samples
import umap
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from amlvae.models.VAE import VAE 
from amlvae.train.Trainer import Trainer

from amlvae.data.ExprProcessor import ExprProcessor
from amlvae.data.ClinProcessor import ClinProcessor

from sklearn.model_selection import KFold

from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split

import umap 
from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor

from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.metrics import roc_auc_score, accuracy_score

# auto reimport 
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
model = torch.load('./data/amlmds_vae_model.pt', map_location=torch.device('cpu'))
aml_clin = pd.read_csv('../data/beataml_clinical_for_inputs.csv')
priorMDS_ids = aml_clin[lambda x: x.priorMDS == 'y'].gdc_id.unique()
drug_all = pd.read_csv('../data/beataml_probit_curve_fits_v4_distr.txt', sep='\t').merge(aml_clin[['gdc_id', 'labId']], left_on='lab_id', right_on='labId', how='inner') 

zdf = pd.read_csv('./data/amlmds_vae_zdf.csv')

drug_names = drug_all.inhibitor.value_counts().reset_index()[lambda x: x['count'] > 300].inhibitor.unique()

/tmp/ipykernel_13282/2338242532.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('./data/amlmds_vae_model.pt', map_location=torch.device('cpu'))


In [23]:
def run_drug_response_prediction(zdf, model, priorMDS_ids): 
    zdf = zdf[lambda x: x['auc'].notna()]

    z_test = zdf[lambda x: x.gdc_id.isin(priorMDS_ids)][zdf.columns[:model.latent_dim]].values 
    y_test = zdf[lambda x: x.gdc_id.isin(priorMDS_ids)]['auc'].values 

    z_train = zdf[~zdf.gdc_id.isin(priorMDS_ids)][zdf.columns[:model.latent_dim]].values
    y_train = zdf[~zdf.gdc_id.isin(priorMDS_ids)]['auc'].values 

    yt = np.quantile(y_train, 0.5) 

    y_train = y_train < yt
    y_test = y_test < yt 

    mod = RandomForestClassifier(n_estimators=250, max_depth=5)

    scores = cross_val_score(mod, z_train, y_train, cv=3, scoring='roc_auc')
    cv_roc = np.mean(scores)

    mod.fit(z_train, y_train)
    y_pred = mod.predict_proba(z_test)[:, 1]

    test_roc = roc_auc_score(y_test, y_pred)
    test_acc = accuracy_score(y_test, y_pred > 0.5)
    return cv_roc, test_roc, test_acc 

In [27]:
res = {'inhibitor': [], 'cv_roc': [], 'test_roc': [], 'test_acc': [], 'nobs':[]}
for i,drug_name in enumerate( drug_names ):
    print(f'progress... {i}/{len(drug_names)}: {drug_name}', end='\r')
    drug = drug_all[['gdc_id', 'inhibitor', 'auc']]
    drug = drug[lambda x: x.inhibitor == drug_name]
    zdf_ = zdf.merge(drug, left_on='id', right_on='gdc_id', how='left')

    cv_roc, test_roc, test_acc = run_drug_response_prediction(zdf_, model, priorMDS_ids)
    
    res['inhibitor'].append(drug_name)
    res['cv_roc'].append(cv_roc)
    res['test_roc'].append(test_roc)
    res['test_acc'].append(test_acc)    
    res['nobs'].append(len(drug.auc.dropna()))

res = pd.DataFrame(res)
res = res.sort_values('test_roc', ascending=False) 
res.head() 
 



,inhibitor,cv_roc,test_roc,test_acc,nobs
86,Idelalisib,0.662502,0.873913,0.727273,374
52,PD173955,0.554701,0.872180,0.727273,384
88,GW-2580,0.630984,0.870370,0.878788,371
62,17-AAG (Tanespimycin),0.770948,0.861111,0.818182,383
81,PRT062607,0.600877,0.840909,0.812500,378


In [31]:
res.head(25) 

,inhibitor,cv_roc,test_roc,test_acc,nobs
86,Idelalisib,0.662502,0.873913,0.727273,374
52,PD173955,0.554701,0.872180,0.727273,384
88,GW-2580,0.630984,0.870370,0.878788,371
62,17-AAG (Tanespimycin),0.770948,0.861111,0.818182,383
81,PRT062607,0.600877,0.840909,0.812500,378
77,Tivozanib (AV-951),0.671157,0.840741,0.727273,379
5,Trametinib (GSK1120212),0.790694,0.835913,0.750000,411
8,Axitinib (AG-013736),0.674449,0.830357,0.722222,400
97,SCH-772984,0.702294,0.828283,0.709677,323
39,Selumetinib (AZD6244),0.794515,0.822917,0.735294,386
